# 3 · Composite simulation-based inference (SBI)

This is abbot's lead differentiator — the case that does **not** fit a plain
BO/DataFrame tool. Each experiment doesn't return a scalar directly; it returns an
*observation*, from which we infer a **posterior over latent parameters** `theta`,
and the objective `F` is a property of that posterior (e.g. "how much posterior
mass lands in a target band").

abbot wires this with `sbi_evaluate(posterior_sampler, reducer, spec)`: it maps
observation → posterior samples over `theta` → a scalar `F`, and caches the `theta`
samples on each record so an annealed objective can be re-scored without
re-sampling.

In [1]:
import torch, warnings
warnings.filterwarnings("ignore")
from abbot import (
    ProblemSpec, ContinuousDomain, OutputSpec, Box,
    Campaign, PosteriorSampler, Rescore, sbi_evaluate,
)
torch.manual_seed(0)

### A toy posterior sampler
In a real SBI app this wraps a trained neural density estimator (e.g. an `sbi`
`DirectPosterior`). Here it's a closed-form Gaussian around the observed value, so
the notebook needs no trained model. The design `x` controls the latent `theta*`;
we want the posterior over `theta` to sit inside the band `[4, 6]`.

In [2]:
class ToyExperiment:
    def run(self, design):                    # design = {"x": 0.42}
        return torch.tensor([design["x"] * 10.0])   # observation centered at theta* = 10*x

class ToyPosterior(PosteriorSampler):
    def sample(self, observation, n=512):
        mean = observation.reshape(-1)
        return mean + 0.5 * torch.randn(n, mean.shape[0])  # posterior over theta

spec = ProblemSpec(
    domain=[ContinuousDomain(name="x", lower=0.0, upper=1.0)],
    outputs=[OutputSpec(name="F", direction="MAXIMIZE")],
    boxes=[Box(lower=[4.0], upper=[6.0])],   # target band on theta
)

### The composite objective
The reducer is yours: it turns a posterior into the number you want to maximize.
Here it is the (soft) fraction of `theta` samples inside the target band, with
`beta` setting how sharp the band edges are. `sbi_evaluate` chains sampler →
reducer and returns `(F, theta)` so `theta` is cached as `aux`.

In [3]:
def reducer(theta, spec, beta=5.0):
    lo, hi = spec.boxes[0].lower[0], spec.boxes[0].upper[0]
    return (torch.sigmoid(beta * (theta - lo)) * torch.sigmoid(beta * (hi - theta))).mean()

evaluate = sbi_evaluate(ToyPosterior(), reducer, spec, n_samples=256)

campaign = Campaign(spec)  # SimpleBO + single-output GP, inferred from the spec
campaign.seed(torch.tensor([[0.05], [0.5], [0.95]], dtype=torch.double))
for _ in range(8):
    campaign.step(ToyExperiment(), evaluate)

In [4]:
rec = campaign.recommend()
best = campaign.history.records[rec[0]]
print("recommended x:", round(float(best.candidate), 3), "(theta* lands in [4,6] near x=0.5)")
print("cached theta samples per record (aux):", campaign.history.records[0].aux.shape)

recommended x: 0.5 (theta* lands in [4,6] near x=0.5)
cached theta samples per record (aux): torch.Size([256, 1])


### Re-scoring from cached posteriors
Because `theta` is cached, the objective can be re-reduced — here with a sharper
`beta` — without re-running the (expensive) inference. `Rescore(reducer, schedule)`
does this; passed as `Campaign(spec, targets=Rescore(...))` it re-scores every
clean shot at each step, so the model, the proposals and the hand-over follow an
annealed objective.

In [5]:
sharp = Rescore(reducer, lambda t: {"beta": 50.0})(campaign.history, spec)
print("re-scored F (sharper beta), first 5:", [round(v, 3) for v in sharp.reshape(-1)[:5].tolist()])

re-scored F (sharper beta), first 5: [0.0, 0.947, 0.0, 0.426, 0.663]


**What this showed:** the objective is a functional of an inferred posterior, not a
direct measurement — and the posterior samples are cached so the objective can be
re-annealed cheaply. This composite-SBI pattern is the thing abbot owns that a
tabular BO tool structurally cannot represent.